### Creating an MCP Server and MCP Client

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [ ]:
from accounts import Account


In [7]:
account = Account.get("Nober")
account

Account(name='nober', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [11]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "nober", "balance": 9510.022, "strategy": "", "holdings": {"AMZN": 12}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 18.036, "timestamp": "2026-03-25 09:26:46", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 70.14, "timestamp": "2026-03-25 09:30:30", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 25.05, "timestamp": "2026-03-25 09:30:45", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 50.1, "timestamp": "2026-03-25 09:30:46", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-03-25 09:26:46", 10011.892], ["2026-03-25 09:30:31", 10203.472], ["2026-03-25 09:30:45", 10173.322], ["2026-03-25 09:30:46", 9918.022]], "total_portfolio_value": 9918.022, "total_profit_loss": -81.97799999999916}'

In [12]:
account.report()

'{"name": "nober", "balance": 9510.022, "strategy": "", "holdings": {"AMZN": 12}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 18.036, "timestamp": "2026-03-25 09:26:46", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 70.14, "timestamp": "2026-03-25 09:30:30", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 25.05, "timestamp": "2026-03-25 09:30:45", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 50.1, "timestamp": "2026-03-25 09:30:46", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-03-25 09:26:46", 10011.892], ["2026-03-25 09:30:31", 10203.472], ["2026-03-25 09:30:45", 10173.322], ["2026-03-25 09:30:46", 9918.022], ["2026-03-25 09:31:49", 10530.022]], "total_portfolio_value": 10530.022, "total_profit_loss": 530.0220000000008}'

In [13]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 18.036,
  'timestamp': '2026-03-25 09:26:46',
  'rationale': 'Because this bookstore website looks promising'},
 {'symbol': 'AMZN',
  'quantity': 3,
  'price': 70.14,
  'timestamp': '2026-03-25 09:30:30',
  'rationale': 'Because this bookstore website looks promising'},
 {'symbol': 'AMZN',
  'quantity': 3,
  'price': 25.05,
  'timestamp': '2026-03-25 09:30:45',
  'rationale': 'Because this bookstore website looks promising'},
 {'symbol': 'AMZN',
  'quantity': 3,
  'price': 50.1,
  'timestamp': '2026-03-25 09:30:46',
  'rationale': 'Because this bookstore website looks promising'}]

In [14]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [15]:
mcp_tools

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\nArgs:\n    name: The name of the account holder\n', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\nArgs:\n    name: The name of the account holder\n', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='buy_s

In [23]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is nober and my account is under the name nober. What's my balance?"
model = "gpt-4.1-mini"

In [24]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[server])
    with trace("investigate"):
        result = await Runner.run(agent, request)
    print(result.final_output)


The cash balance in your account under the name "nober" is $9,510.02. Is there anything else you would like to check or do with your account?


In [25]:
# Build a simple MCP client

from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\nArgs:\n    name: The name of the account holder\n', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None), Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\nArgs:\n    name: The name of the account holder\n', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None), Tool(name='buy_sha

In [26]:
request = "My name is Ed and my account is under the name Ed. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Ed, your account balance is $10,000.00. Is there anything specific you would like to do with your account today?

In [27]:
context = await read_accounts_resource("ed")
print(context)

{"name": "ed", "balance": 10000.0, "strategy": "", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-03-25 10:01:41", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}


In [28]:
from accounts import Account
Account.get("ed").report()

'{"name": "ed", "balance": 10000.0, "strategy": "", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-03-25 10:01:41", 10000.0], ["2026-03-25 10:02:14", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}'

### Exercise to be worked on..